In [2]:
import pandas as pd

quarter_map = {
    "Quarter 1": 1,
    "Quarter 2": 2,
    "Quarter 3": 3,
    "Quarter 4": 4
}

In [3]:
real_estate = pd.read_csv('../data/raw/61262-0002_en_flat.csv',
                 usecols=['time', '1_variable_attribute_label',
                          'value', 'value_variable_code'],
                 sep=';')
stocks = pd.read_csv('../data/raw/stocks_raw.csv', index_col=0)

print(real_estate.head())
print(stocks.head())

   time 1_variable_attribute_label  value value_variable_code
0  2021                  Quarter 1  148.7              PRE038
1  2021                  Quarter 1  133.3              PRE037
2  2021                  Quarter 1  146.3              PRE026
3  2020                  Quarter 1  135.3              PRE038
4  2020                  Quarter 1  126.0              PRE037
         Date    EUNL.DE        ^GSPC
0  2016-01-04  37.029999  2012.660034
1  2016-01-05  37.570000  2016.709961
2  2016-01-06  37.090000  1990.260010
3  2016-01-07  36.119999  1943.089966
4  2016-01-08  35.380001  1922.030029


In [5]:
real_estate['quarter_num'] = real_estate['1_variable_attribute_label'].map(quarter_map)

real_estate['period'] = pd.PeriodIndex.from_fields(
    year=real_estate['time'],
    quarter=real_estate['quarter_num'],
    freq='Q'
)

re_pivot = real_estate.pivot_table(
    index='period',
    columns='value_variable_code',
    values='value'
).reset_index()

re_pivot.columns.name = None
re_pivot = re_pivot.rename(columns={
    'PRE026': 'house_price_index',
    'PRE037': 'new_construction_index',
    'PRE038': 'existing_property_index'
})

print(re_pivot.head())

   period  house_price_index  new_construction_index  existing_property_index
0  2016Q1              103.9                   104.2                    103.8
1  2016Q2              106.9                   106.2                    107.0
2  2016Q3              108.8                   107.5                    109.0
3  2016Q4              110.4                   109.5                    110.5
4  2017Q1              110.9                   109.3                    111.2


In [6]:
stocks['Date'] = pd.to_datetime(stocks['Date'])
stocks = stocks.set_index('Date')

stocks_q = stocks.resample('QE').last().reset_index()
stocks_q['period'] = stocks_q['Date'].dt.to_period('Q')
stocks_q = stocks_q.drop(columns='Date')

print(stocks_q.head())

     EUNL.DE        ^GSPC  period
0  35.990002  2059.739990  2016Q1
1  37.139999  2098.860107  2016Q2
2  38.709999  2168.270020  2016Q3
3  42.180000  2238.830078  2016Q4
4  44.070000  2362.719971  2017Q1


In [7]:
merged = pd.merge(stocks_q, re_pivot, on='period', how='inner')

cols_to_normalize = ['EUNL.DE', '^GSPC', 'house_price_index',
                     'new_construction_index', 'existing_property_index']

for col in cols_to_normalize:
    merged[f'{col}_norm'] = (merged[col] / merged[col].iloc[0]) * 100

print(merged.head())

     EUNL.DE        ^GSPC  period  house_price_index  new_construction_index  \
0  35.990002  2059.739990  2016Q1              103.9                   104.2   
1  37.139999  2098.860107  2016Q2              106.9                   106.2   
2  38.709999  2168.270020  2016Q3              108.8                   107.5   
3  42.180000  2238.830078  2016Q4              110.4                   109.5   
4  44.070000  2362.719971  2017Q1              110.9                   109.3   

   existing_property_index  EUNL.DE_norm  ^GSPC_norm  house_price_index_norm  \
0                    103.8    100.000000  100.000000              100.000000   
1                    107.0    103.195326  101.899275              102.887392   
2                    109.0    107.557647  105.269113              104.716073   
3                    110.5    117.199217  108.694791              106.256015   
4                    111.2    122.450674  114.709623              106.737247   

   new_construction_index_norm  existi

In [8]:
merged.to_csv('../data/processed/01_roi_comparison.csv', index=False)